# A2 — Des films qui se ressemblent

| Projet | Étape | Durée | Modules réinvestis |
|---|---|---|---|
| A — Recommandation de films | 2 sur 4 | 3 h | M1, M5, M6, M8 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Le modèle des biais de l'étape A1 recommande les mêmes films à tout le monde. Pour **personnaliser**, une idée simple vient des librairies et des sites marchands : « les personnes qui ont aimé ce film ont aussi aimé… ». Deux films se ressemblent si les mêmes spectateurs les apprécient (ou les rejettent) ensemble.

**La sous-question de cette étape : comment mesurer la ressemblance entre deux films à partir des seules notes, et comment s'en servir pour prévoir la note d'un spectateur ?**

**Activité** : Au module M1, vous avez comparé des recettes avec la distance et avec le cosinus. Lequel de ces outils semble adapté ici ? Pourquoi faut-il d'abord retirer les biais des notes avant de comparer deux films ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_gain_reqm = 0.05        # baisse de l'erreur de validation par rapport au modèle des biais, en étoiles (modifiez)
pari_voisins_meme_genre = 0.5    # part des 10 voisins d'un film qui partagent au moins un de ses genres (modifiez)

## ③ Abstraire : chaque film est un vecteur

On calcule pour chaque note connue l'**écart** au modèle des biais : $e_{uf} = r_{uf} - (\mu + b_u + b_f)$. Un écart positif signifie « ce spectateur a plus aimé ce film que prévu ». Pour un spectateur qui n'a pas vu le film, on pose $e_{uf} = 0$ : aucune information.

Chaque film devient alors un vecteur colonne de la matrice des écarts $E$, avec une coordonnée par spectateur. La ressemblance entre deux films $f$ et $g$ est le **cosinus** de l'angle entre leurs vecteurs (module M1) :

$$
s_{fg} = \frac{\mathbf{e}_f \cdot \mathbf{e}_g}{\lVert \mathbf{e}_f \rVert\,\lVert \mathbf{e}_g \rVert} .
$$

Toutes les similarités s'obtiennent d'un coup par un produit de matrices : $E^{\mathsf{T}}E$, divisé par les normes.

**Méfiance pour les petits effectifs.** Deux films vus par **un seul** spectateur commun, qui les a aimés tous les deux, ont un cosinus proche de 1 : c'est une coïncidence, pas une ressemblance. On atténue la similarité selon le nombre $n_{fg}$ de spectateurs communs :

$$
s'_{fg} = s_{fg} \times \frac{n_{fg}}{n_{fg} + \beta} .
$$

**Prévoir.** Pour estimer la note du spectateur $u$ pour le film $f$, on regarde les $k$ films **qu'il a notés** et qui ressemblent le plus à $f$, et l'on fait la moyenne de ses écarts sur ces films, pondérée par la ressemblance :

$$
\hat r_{uf} = \mu + b_u + b_f + \frac{\sum_{g \in \text{voisins}} s'_{fg}\, e_{ug}}{\sum_{g \in \text{voisins}} \lvert s'_{fg} \rvert} .
$$

**Activité** : Trois spectateurs ont pour écarts $(1\,;\,-1\,;\,0{,}5)$ sur le film A et $(0{,}8\,;\,-0{,}6\,;\,0)$ sur le film B. Calculez le cosinus entre A et B. Un quatrième spectateur a un écart de $+1$ sur B : que prévoyez-vous pour lui sur A, si B est le seul voisin ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Préparer la matrice des écarts

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from films import charger_films, decouper, reqm, Index, ajuster_biais, predire_biais

notes, films, source = charger_films()
reste, test = decouper(notes, part_test=0.2, graine=0)      # mêmes paquets qu'en A1 ; le test reste sous scellés
app, val = decouper(reste, part_test=0.2, graine=1)
modele = ajuster_biais(app, lam=10)
mu, bu, bf = modele

# On compare les films assez notés (au moins 10 notes) ; les autres garderont la prédiction des biais
nb_notes_film = app.groupby("movieId").size()
app_k = app[app["movieId"].isin(nb_notes_film[nb_notes_film >= 10].index)]
idx = Index(app_k)
R = idx.matrice(app_k)
connu = ~np.isnan(R)
base = mu + np.array([bu.get(u, 0) for u in idx.spectateurs])[:, None] + np.array([bf.get(f, 0) for f in idx.films])[None, :]
E = np.where(connu, R - base, 0.0)
print(f"Matrice des écarts : {E.shape[0]} spectateurs × {E.shape[1]} films, {connu.mean():.1%} de cases connues")

### Toutes les similarités d'un coup

In [ ]:
def similarites(E, connu, beta=25):
    normes = np.linalg.norm(E, axis=0) + 1e-12
    S = (E.T @ E) / np.outer(normes, normes)                 # cosinus entre colonnes
    communs = connu.T.astype(float) @ connu.astype(float)     # nombre de spectateurs communs
    S = S * communs / (communs + beta + 1e-12)                # atténuation des petits effectifs
    np.fill_diagonal(S, 0)
    return S, communs

debut = time.perf_counter()
S, communs = similarites(E, connu)
print(f"{S.shape[0]} × {S.shape[1]} similarités en {time.perf_counter() - debut:.2f} s")

### Les voisins d'un film

In [ ]:
infos = films.set_index("movieId")
def voisins(film_id, S, k=8):
    j = idx.colonne[film_id]
    ordre = np.argsort(-S[j])[:k]
    return pd.DataFrame({"titre": [infos.loc[idx.films[o], "title"] for o in ordre],
                         "genres": [infos.loc[idx.films[o], "genres"] for o in ordre],
                         "similarité": S[j, ordre].round(3),
                         "spectateurs communs": communs[j, ordre].astype(int)})

populaire = nb_notes_film.idxmax()
print("Film choisi :", infos.loc[populaire, "title"], "—", infos.loc[populaire, "genres"])
voisins(populaire, S)

**Activité** : Ces voisins vous semblent-ils pertinents ? Que pouvez-vous en dire, sans avoir vu les films ? Comment vérifier, de façon chiffrée, que les voisins ont « quelque chose en commun » avec le film choisi ?

Indiquez vos réponses et justifiez-les.

### Corroborer : les voisins partagent-ils des genres ?

Les genres n'ont **pas** été utilisés pour calculer les similarités. S'ils se retrouvent chez les voisins plus souvent que par hasard, c'est un indice que la méthode capte quelque chose de réel.

In [ ]:
genres = {f: set(infos.loc[f, "genres"].split("|")) for f in idx.films}

def part_meme_genre(S, k=10, echantillon=300, graine=0):
    rng = np.random.default_rng(graine)
    choisis = rng.choice(len(idx.films), echantillon, replace=False)
    voisins_ok, hasard_ok = [], []
    for j in choisis:
        g = genres[idx.films[j]]
        for o in np.argsort(-S[j])[:k]:
            voisins_ok.append(bool(g & genres[idx.films[o]]))
        for o in rng.choice(len(idx.films), k):
            hasard_ok.append(bool(g & genres[idx.films[o]]))
    return np.mean(voisins_ok), np.mean(hasard_ok)

voisins_genre, hasard_genre = part_meme_genre(S)
print(f"Voisins qui partagent un genre : {voisins_genre:.0%} ; films tirés au hasard : {hasard_genre:.0%}")

### L'effet de l'atténuation sur un film peu noté

In [ ]:
S_brut, _ = similarites(E, connu, beta=0)
peu_note = nb_notes_film[nb_notes_film.index.isin(idx.films)].idxmin()
print("Film peu noté :", infos.loc[peu_note, "title"], f"({nb_notes_film[peu_note]} notes)")
print("\nSans atténuation (β = 0) :")
print(voisins(peu_note, S_brut, 5)[["titre", "similarité", "spectateurs communs"]].to_string(index=False))
print("\nAvec atténuation (β = 25) :")
print(voisins(peu_note, S, 5)[["titre", "similarité", "spectateurs communs"]].to_string(index=False))

### Prévoir les notes de validation

In [ ]:
def predire_voisins(S, notes_a_predire, k=30):
    pred = predire_biais(modele, notes_a_predire)              # par défaut : le modèle des biais
    lignes = [idx.ligne.get(u, -1) for u in notes_a_predire["userId"]]
    colonnes = idx.colonnes(notes_a_predire["movieId"])
    for n, (i, j) in enumerate(zip(lignes, colonnes)):
        if i < 0 or j < 0:
            continue
        vus = np.nonzero(connu[i])[0]
        s = S[j, vus]
        meilleurs = np.argsort(-np.abs(s))[:k]
        poids = s[meilleurs]
        if np.abs(poids).sum() > 0:
            pred[n] = base[i, j] + poids @ E[i, vus[meilleurs]] / np.abs(poids).sum()
    return np.clip(pred, 0.5, 5)

reqm_biais = reqm(val["rating"], predire_biais(modele, val))
print(f"modèle des biais  : {reqm_biais:.4f}")
resultats_k = {}
for k in [5, 10, 20, 40, 80]:
    debut = time.perf_counter()
    resultats_k[k] = reqm(val["rating"], predire_voisins(S, val, k))
    print(f"voisins, k = {k:3d} : {resultats_k[k]:.4f}   ({time.perf_counter() - debut:.1f} s)")

### Recommander à un spectateur

In [ ]:
def recommander_voisins(user_id, S, n=10, k=30):
    i = idx.ligne[user_id]
    candidats = idx.films[~connu[i]]
    a_predire = pd.DataFrame({"userId": user_id, "movieId": candidats})
    a_predire["prévision"] = predire_voisins(S, a_predire, k)
    a_predire["titre"] = a_predire["movieId"].map(infos["title"])
    a_predire["genres"] = a_predire["movieId"].map(infos["genres"])
    return a_predire.sort_values("prévision", ascending=False).head(n)

spectateur = int(app.groupby("userId").size().sort_values().index[len(idx.spectateurs) // 2])   # un spectateur « moyen »
print("Ses films préférés (apprentissage) :")
favoris = app[app["userId"] == spectateur].sort_values("rating", ascending=False).head(5)
print(favoris.assign(titre=favoris["movieId"].map(infos["title"]), genres=favoris["movieId"].map(infos["genres"]))
      [["titre", "genres", "rating"]].to_string(index=False))
print("\nRecommandations :")
print(recommander_voisins(spectateur, S)[["titre", "genres", "prévision"]].round(2).to_string(index=False))

## ⑤ Interpréter

In [ ]:
k_meilleur = min(resultats_k, key=resultats_k.get)
print(f"Vos paris : gain {pari_gain_reqm} étoile ; voisins de même genre {pari_voisins_meme_genre:.0%}")
print(f"Mesures   : gain {reqm_biais - resultats_k[k_meilleur]:.3f} étoile (k = {k_meilleur}) ; voisins de même genre {voisins_genre:.0%}")

**Activité** :

1. La méthode des voisins améliore-t-elle les prévisions ? De combien, en étoiles ? Ce gain vous paraît-il utile pour un spectateur ?
2. Les recommandations du spectateur choisi sont-elles cohérentes avec ses films préférés ? Sur quoi vous appuyez-vous ?
3. Quels traits intéressants avez-vous remarqués : effet de l'atténuation, choix de $k$, films peu notés ?
4. Cette méthode aide-t-elle un spectateur qui n'a noté que trois films ? Et un film que personne n'a encore noté ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Le coût.** La matrice des similarités a autant de lignes et de colonnes que de films. Combien de mémoire faudrait-il pour 50 000 films ? Comment s'en sortir ?
- **D'autres choix.** On a comparé les films ; on pourrait comparer les **spectateurs** (« les spectateurs qui vous ressemblent ont aimé… »). Lequel des deux paraît plus stable dans le temps ? Lequel est plus facile à expliquer à un abonné ?
- **Généraliser.** Écrivez une fonction `predire_par_spectateurs(notes_a_predire, k=30, beta=25)` qui utilise la similarité entre **lignes** de $E$ : pour prévoir la note de $u$ sur $f$, on prend les $k$ spectateurs les plus semblables à $u$ **qui ont noté** $f$. Comparez son erreur de validation à celle des films voisins.

In [ ]:
def predire_par_spectateurs(notes_a_predire, k=30, beta=25):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
S_spect, _ = similarites(E.T, connu.T, beta=25)            # mêmes calculs sur les lignes

def predire_par_spectateurs(notes_a_predire, k=30, beta=25):
    pred = predire_biais(modele, notes_a_predire)
    lignes = [idx.ligne.get(u, -1) for u in notes_a_predire["userId"]]
    colonnes = idx.colonnes(notes_a_predire["movieId"])
    for n, (i, j) in enumerate(zip(lignes, colonnes)):
        if i < 0 or j < 0:
            continue
        ont_vu = np.nonzero(connu[:, j])[0]
        s = S_spect[i, ont_vu]
        meilleurs = np.argsort(-np.abs(s))[:k]
        poids = s[meilleurs]
        if np.abs(poids).sum() > 0:
            pred[n] = base[i, j] + poids @ E[ont_vu[meilleurs], j] / np.abs(poids).sum()
    return np.clip(pred, 0.5, 5)

print(f"spectateurs voisins : {reqm(val['rating'], predire_par_spectateurs(val)):.4f}   "
      f"films voisins : {resultats_k[k_meilleur]:.4f}")
print(f"Mémoire pour 50 000 films : {50_000 ** 2 * 8 / 1e9:.0f} Go ; on ne garde en pratique que les 50 à 100 meilleurs voisins de chaque film.")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Appliquer des outils connus dans des contextes nouveaux": 1,
    "Identifier les concepts pertinents et leurs relations": 1,
    "Corroborer ses résultats": 1,
    "Repérer les traits intéressants du résultat": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "A2-ressemblances", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")